In [ ]:
import pandas as pd
import os

# PADRÃO DE INGESTÃO DE DADOS (ETL - EXTRACT)

# 1. Definição Automática de Caminhos
# O os.getcwd() pega a pasta onde o notebook está, tornando o projeto portável.
BASE_PATH = os.getcwd()
RAW_PATH = os.path.join(BASE_PATH, 'data', 'raw')

print(f"📂 Diretório Base: {BASE_PATH}")
print(f"📂 Buscando dados em: {RAW_PATH}\n")

# 2. Pipeline de Ingestão Dinâmica
# Dicionário 'dfs' para armazenar todas as tabelas organizadas.
dfs = {} 

try:
    # Lista todos os arquivos na pasta
    files = os.listdir(RAW_PATH)
    # Filtra apenas os que terminam com .csv (ignora zips, txts, ocultos)
    csv_files = [f for f in files if f.endswith('.csv')]
    
    if not csv_files:
        print("❌ Erro: Nenhum arquivo CSV encontrado na pasta raw.")
    else:
        print(f"✅ Encontrados {len(csv_files)} arquivos. Iniciando carregamento...\n")
        
        for file in csv_files:
            # LIMPEZA DE NOME: Remove prefixos/sufixos para criar uma chave limpa
            # Ex: 'olist_orders_dataset.csv' vira apenas 'orders'
            name_clean = file.replace('olist_', '').replace('_dataset.csv', '')
            
            # Caminho completo do arquivo
            file_path = os.path.join(RAW_PATH, file)
            
            # Carrega o arquivo para dentro do dicionário
            dfs[name_clean] = pd.read_csv(file_path)
            
            # Log de confirmação (Nome da tabela e tamanho: Linhas x Colunas)
            print(f"   📦 Tabela: {name_clean:<20} | Shape: {dfs[name_clean].shape}")

        print("\n🚀 Carga concluída! Todos os DataFrames estão no dicionário 'dfs'.")
        print("Exemplo de uso: dfs['orders'].head()")
        
except FileNotFoundError:
    print(f"❌ Erro Crítico: A pasta '{RAW_PATH}' não existe.")

📂 Diretório Base: g:\Meu Drive\2026\Projetos dados\Projeto BI OLIST\portfolio_olist
📂 Buscando dados em: g:\Meu Drive\2026\Projetos dados\Projeto BI OLIST\portfolio_olist\data\raw

✅ Encontrados 9 arquivos. Iniciando carregamento...

   📦 Tabela: sellers              | Shape: (3095, 4)
   📦 Tabela: product_category_name_translation.csv | Shape: (71, 2)
   📦 Tabela: order_items          | Shape: (112650, 7)
   📦 Tabela: geolocation          | Shape: (1000163, 5)
   📦 Tabela: products             | Shape: (32951, 9)
   📦 Tabela: orders               | Shape: (99441, 8)
   📦 Tabela: order_reviews        | Shape: (99224, 7)
   📦 Tabela: order_payments       | Shape: (103886, 5)
   📦 Tabela: customers            | Shape: (99441, 5)

🚀 Carga concluída! Todos os DataFrames estão no dicionário 'dfs'.
Exemplo de uso: dfs['orders'].head()


In [6]:
# ==============================================================================
# TRANSFORM - LIMPEZA E ENGENHARIA DE ATRIBUTOS
# ==============================================================================

# 1. Ajuste rápido de nomes (Quality of Life)
if 'product_category_name_translation.csv' in dfs:
    dfs['category_translation'] = dfs.pop('product_category_name_translation.csv')
    print("✅ Nome da tabela de tradução ajustado para 'category_translation'")

# 2. Conversão de Datas (Type Casting)
# O Pandas carrega datas como 'object' (texto) por padrão. Precisamos converter.
print("\n🔄 Convertendo colunas de data na tabela 'orders'...")

colunas_data = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Acessamos a tabela direto do dicionário
df_orders = dfs['orders']

for col in colunas_data:
    df_orders[col] = pd.to_datetime(df_orders[col])

print("✅ Datas convertidas com sucesso!")

# 3. Feature Engineering (Criando métricas de negócio)
print("\n🛠️ Criando novas métricas logísticas...")

# A. Tempo de Entrega Real (Dias)
# Diferença entre: Data que chegou no cliente - Data da compra
df_orders['time_to_delivery'] = (
    df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']
).dt.days

# B. Diferença do Estimado (Dias)
# Diferença entre: Data Estimada - Data Real (Positivo = Chegou antes, Negativo = Atrasou)
df_orders['diff_estimated_delivery'] = (
    df_orders['order_estimated_delivery_date'] - df_orders['order_delivered_customer_date']
).dt.days

# C. Flag de Atraso (KPI Binário: 1 ou 0)
# Lógica: Se a data real for MAIOR que a estimada, é atraso.
df_orders['is_late'] = df_orders['order_delivered_customer_date'] > df_orders['order_estimated_delivery_date']
df_orders['is_late'] = df_orders['is_late'].astype(int) # Converte True/False para 1/0

print("✅ Métricas criadas: 'time_to_delivery', 'diff_estimated_delivery', 'is_late'")

# 4. Verificação Final
print("\n📊 Amostra das novas colunas:")
print(df_orders[['order_purchase_timestamp', 'time_to_delivery', 'is_late']].head())

✅ Nome da tabela de tradução ajustado para 'category_translation'

🔄 Convertendo colunas de data na tabela 'orders'...
✅ Datas convertidas com sucesso!

🛠️ Criando novas métricas logísticas...
✅ Métricas criadas: 'time_to_delivery', 'diff_estimated_delivery', 'is_late'

📊 Amostra das novas colunas:
  order_purchase_timestamp  time_to_delivery  is_late
0      2017-10-02 10:56:33               8.0        0
1      2018-07-24 20:41:37              13.0        0
2      2018-08-08 08:38:49               9.0        0
3      2017-11-18 19:28:06              13.0        0
4      2018-02-13 21:18:39               2.0        0


In [ ]:

# MERGE (TRANSFORM) - APENAS A UNIFICAÇÃO

print("\n🔄 Iniciando unificação das tabelas (Join)...")

# 1. Preparação das Tabelas
# Garantimos que estamos pegando as versões atualizadas do dicionário
df_orders = dfs['orders'].copy()
df_items = dfs['order_items'].copy()

# 2. O Grande Merge
# Unimos Pedidos (que tem as datas) com Itens (que tem o preço)
# Lógica: 'inner' join garante que só pegamos pedidos que realmente têm itens.
fact_vendas = pd.merge(
    left=df_items,           # Tabela Detalhe (Dinheiro/Produto)
    right=df_orders,         # Tabela Cabeçalho (Datas/Cliente/Status)
    on='order_id',           # A chave de ligação
    how='inner'
)

print(f"✅ Merge concluído!")
print(f"   - Linhas na tabela Items (Origem): {df_items.shape[0]}")
print(f"   - Linhas na Fato Vendas (Final):   {fact_vendas.shape[0]}")

# Validação Rápida: Se o número de linhas da Fato for igual ao de Items, o merge foi perfeito (1:1 na granularidade do item).
if df_items.shape[0] == fact_vendas.shape[0]:
    print("   ✨ Validação: O número de linhas bateu perfeitamente!")
else:
    print("   ⚠️ Atenção: Houve perda ou multiplicação de linhas. Verifique se há pedidos sem itens.")

# 3. Limpeza Final na Fato
# Removemos colunas que não vamos usar no BI para deixar o arquivo leve
cols_to_drop = ['order_approved_at', 'order_delivered_carrier_date'] 
fact_vendas = fact_vendas.drop(columns=cols_to_drop, errors='ignore')


🔄 Iniciando unificação das tabelas (Join)...
✅ Merge concluído!
   - Linhas na tabela Items (Origem): 112650
   - Linhas na Fato Vendas (Final):   112650
   ✨ Validação: O número de linhas bateu perfeitamente!


In [10]:
# ==============================================================================
# 4. SALVAMENTO (LOAD)
# ==============================================================================
import os
import pandas as pd

# --- RECALCULANDO CAMINHOS (Para garantir que funcione isolado) ---
base_path = os.getcwd() 
processed_dir = os.path.join(base_path, 'data', 'processed')
# ------------------------------------------------------------------

os.makedirs(processed_dir, exist_ok=True) 

print(f"📂 Salvando arquivos em: {processed_dir}")

try:
    # Salva a Tabela Fato
    if 'fact_vendas' in locals():
        fact_vendas.to_csv(os.path.join(processed_dir, 'fact_vendas.csv'), index=False)
        print("   ✅ fact_vendas.csv salvo!")
    else:
        print("   ❌ Erro: A tabela 'fact_vendas' não existe na memória. Rode a célula de Merge anterior.")

    # Salva as Dimensões
    dimensoes = ['products', 'customers', 'sellers', 'geolocation', 'category_translation']
    for dim in dimensoes:
        if 'dfs' in locals() and dim in dfs:
            dfs[dim].to_csv(os.path.join(processed_dir, f'dim_{dim}.csv'), index=False)
            print(f"   ✅ dim_{dim}.csv salvo!")
            
    print("\n🎉 ARQUIVOS GERADOS COM SUCESSO!")

except Exception as e:
    print(f"Erro: {e}")

📂 Salvando arquivos em: g:\Meu Drive\2026\Projetos dados\Projeto BI OLIST\portfolio_olist\data\processed
   ✅ fact_vendas.csv salvo!
   ✅ dim_products.csv salvo!
   ✅ dim_customers.csv salvo!
   ✅ dim_sellers.csv salvo!
   ✅ dim_geolocation.csv salvo!
   ✅ dim_category_translation.csv salvo!

🎉 ARQUIVOS GERADOS COM SUCESSO!
